In [3]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [4]:
df=datasets.load_diabetes(as_frame=True).frame

In [5]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [6]:
df.shape

(442, 11)

In [7]:
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

In [8]:
X=df.drop("target",axis=1)
y=df["target"]

In [9]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [10]:
y.head()

0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64

In [11]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.3,
    random_state=42  
)

In [12]:
y_scaler=StandardScaler()

y_train_scaled=y_scaler.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled=y_scaler.transform(y_test.values.reshape(-1,1)).ravel()

In [14]:
model=SVR()

model.fit(X_train,y_train_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [15]:
y_pred_scaled=model.predict(X_test)

In [16]:
print(r2_score(y_test_scaled,y_pred_scaled))

0.48844443151651884


In [18]:
# linear kernel
model=SVR(kernel="linear")

model.fit(X_train,y_train_scaled)
y_pred_scaled=model.predict(X_test)
print(r2_score(y_test_scaled,y_pred_scaled))

0.4433761323833776


In [19]:
# poly kernel
model=SVR(kernel="poly")

model.fit(X_train,y_train_scaled)
y_pred_scaled=model.predict(X_test)
print(r2_score(y_test_scaled,y_pred_scaled))

0.24203771038107758


# Hyper parameter tuning using gridsearchcv

In [21]:
from sklearn.model_selection import GridSearchCV

In [25]:
param_grid={
    "C":[1,2,5,10,50,100],
    "kernel":["rbf","linear","poly"],
    "epsilon":[0.01,0.1,0.2,0.3,]
}

In [27]:
svr=SVR()

grid_search=GridSearchCV(svr,param_grid,scoring="r2",cv=5)

grid_search.fit(X_train,y_train_scaled)

,estimator,SVR()
,param_grid,"{'C': [1, 2, ...], 'epsilon': [0.01, 0.1, ...], 'kernel': ['rbf', 'linear', ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'linear'


In [28]:
print("besr params = ",grid_search.best_params_)

besr params =  {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [29]:
best_model=SVR(kernel="linear",C=10,epsilon=0.1)
best_model.fit(X_train,y_train_scaled)
y_pred_scaled=best_model.predict(X_test)
print(r2_score(y_test_scaled,y_pred_scaled))

0.47444183250401095
